# 04b. Graph Engineering ②: 실패별 다음 행동, 부분 실패, 승인과 멱등

04a에서 Worker가 주고받는 계약(Prompt, Context)과 공유 예산(Harness)을 그래프에 넣었습니다. 이 노트북 **04b**는 나머지 두 가지를 넣습니다. 실패했을 때 무엇을 할지(Graph와 Loop), 환불 같은 쓰기를 언제 실행할지(승인과 멱등)입니다. 사례는 계속 고객 요청 T-1001입니다. "결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요."

## 학습 목표

- 실패 종류에 따라 다음 행동을 다르게 정하고, `RetryPolicy`와 `next_action`이 맡는 범위가 다름을 확인합니다.
- 병렬 Worker의 부분 실패를 합류 노드에서 드러내고, 완료 결과를 보존한 채 실패한 작업만 다시 돌립니다.
- 승인을 실행 앞에 두고, 재개와 재시도에서 중복 실행이 어떻게 생기고 어떻게 막히는지 실행해 봅니다.
- 앞 절의 부품을 한 그래프에 모으고, 코드·모델·사람이 맡는 노드를 구분합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 5 | Graph와 Loop: 실패별 다음 행동 | 8 |
| 6 | 부분 실패와 합류 | 5 |
| 7 | 승인과 멱등 | 9 |
| 8 | 완성형 조립 | 5 |
| 9 | 직접 해보기와 정리 | 3 |

## 호출 규모

이 노트북은 `Day03_new/.env`의 키로 실제 Claude 모델을 부릅니다. 키 없이 도는 경로는 없습니다. 모델을 부르는 셀은 호출 직전에 `announce`가 규모를 한 줄로 알려 줍니다. 비용은 본인 키로 청구됩니다. 실패 분류, 합류, 게이트, 멱등처럼 모델의 판단이 필요 없는 부분은 일반 함수로 돌리고 출력에 "모델 호출 0회"를 적습니다. 모델이 할 일이 없는 곳에서 모델을 부르지 않기 위해서입니다.

| 구분 | 유료 셀 수 | 모델 호출(예고 상한) |
|---|---:|---:|
| 5절 시간 초과 1회 주입 뒤 Worker 복구 | 1 | 12 |
| 7절 모델이 T-1001 환불 제안을 작성 | 1 | 5 |
| 8절 완성형 조립으로 T-1001 처리 | 1 | 16 |
| 기본 실행 합계 | 3 | 예고 상한 33 |

`직접 해보기`의 두 칸은 모델을 부르지 않습니다. 호출 수는 실행마다 달라집니다. 모델이 도구를 몇 번 쓰는지는 모델이 정하기 때문입니다.

In [ ]:
import inspect
import json
from collections import Counter
from pathlib import Path
from typing import Annotated, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, Send, interrupt

from mas_lab.contracts import (FailureKind, FinalAnswer, Proposal, WorkerResult, WorkerTask, classify_failure, join_results,
                               merge_results, next_action)
from mas_lab.desk import Desk
from mas_lab.evaluation import evaluate_row, load_gold
from mas_lab.hybrid import ROLE_GOALS, ROLES, build_hybrid
from mas_lab.ledger import GateDecision, PaymentLedger, ResponseLost, check_refund, execute_approved
from mas_lab.runner import make_desk, run_case
from mas_lab.structures import scope_line
from mas_lab.trace import ModelMeter, Trace, timeline
from mas_lab.view import show_graph, show_table, write_design_log
from mas_lab.workers import make_worker, run_worker
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

require_key()  # 키가 없으면 여기서 한국어 안내와 함께 멈춥니다.
print(key_status())
model = make_model()  # 모델 객체를 만드는 것은 호출이 아닙니다.
cases = [json.loads(line) for line in Path("data/cases.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
case = next(c for c in cases if c["id"] == "complex-double-pay")
gold = load_gold()  # 사후 자동 검사에만 씁니다. 도구, 프롬프트, 게이트는 이 값을 보지 않습니다.
print("관통 사례:", case["id"], "|", case["question"])
print("측정 장치:", Desk.__name__, Trace.__name__, ModelMeter.__name__)
# 7절의 멱등 재현은 결제 목록과 기준일만 읽습니다. 모델도 도구도 부르지 않습니다.
case_desk = make_desk(case, label="ledger")

## 04a에서 이어 쓰는 값

5~8절은 04a에서 만든 값을 새로 쓰지 않습니다. 이 노트북은 04a의 kernel 없이도 돌아야 하므로, 04a의 첫 코드 셀과 같은 준비(키 확인, 모델, T-1001 사례, 정답표)를 아래에서 다시 합니다. 새로 배우는 내용은 없습니다. 이 노트북에서 04a의 부품을 부르는 곳은 5절의 `next_action`, 6절의 `join_results`, 8절의 `build_hybrid` 같은 이름뿐이고, 계약과 예산의 뜻은 04a의 1~4절에 있습니다.

## 5. Graph와 Loop: 실패했다고 무조건 다시 부르지 않습니다

**문제.** 가장 흔한 복구 코드는 `try`로 잡고 다시 호출하는 것입니다. 시간 초과라면 맞을 수 있습니다. 그런데 근거가 부족한 답을 그대로 다시 호출하면 비슷한 답에 비용만 더 듭니다. 정책상 환불이 불가능한 요청을 다시 호출하면 같은 거절이 나오거나, 더 나쁘게는 다른 경로로 뚫으려는 시도가 됩니다. 예산을 넘었는데 다시 부르면 예산이 뜻을 잃습니다.

실패의 **종류**가 다음 행동을 정합니다. 이 분류는 모델이 아니라 코드가 합니다.

**먼저 예측하세요.** 다음 표의 `다음 행동`을 가리고 근거 부족, 정책 불가, 시스템 오류, 예산 초과 각각의 다음 행동을 먼저 적으세요. 재시도 기회가 남았을 때와 다 썼을 때를 나눠 적습니다.

In [ ]:
print(inspect.getsource(classify_failure))
print(inspect.getsource(next_action))
show_table([{"실패 종류": kind.value, "재시도 남음": retries, "예산 남음": left, "다음 행동": next_action(kind, retries_left=retries, budget_left=left)}
            for kind in FailureKind for retries, left in ((1, True), (0, True), (1, False))])
print("모델 호출 0회")

### 그림으로 보기: 실패 종류에 따른 다음 행동

위 표의 값(`next_action`을 직접 불러 얻은 값)을 한 장으로 줄였습니다. 칸마다 `next_action`이 고르는 행동이고, 강조한 칸이 다시 부르는 행동입니다.

![실패 종류 다섯 가지와 세 조건(재시도 남음·예산 남음, 재시도 0, 예산 없음)에 따른 next_action의 값. 다시 부르는 칸은 insufficient의 reinvestigate와 system_error의 retry 두 곳뿐이다](./assets/04b-1-failure-actions.png)

- 다시 부르는 칸은 두 곳뿐입니다. 근거가 부족하면 다시 조사하고, 시스템 오류면 같은 작업을 다시 돌립니다.
- 정책상 불가와 예산 초과는 재시도 기회가 남아도 다시 부르지 않습니다. 같은 거절을 반복하거나 예산의 뜻을 없애기 때문입니다.
- 재시도 기회를 다 쓰거나 예산이 없으면 시스템 오류는 사람에게 넘기고(`escalate`), 근거 부족은 이유를 적고 멈춥니다.

In [ ]:
class FailState(TypedDict, total=False):
    step: int
    retries_left: int
    kind: str
    action: str
    path: list[str]

def build_failure_graph(act, decide=next_action, retries: int = 1):
    # act(step)가 (오류, 최종 답, 문제 목록)을 돌려줍니다. 분류와 다음 행동은 코드가 정합니다.
    def attempt(state: FailState) -> dict:
        error, final, problems = act(state.get("step", 0))
        kind = classify_failure(error, final, problems)
        action = decide(kind, retries_left=state["retries_left"], budget_left=not isinstance(error, BudgetExceeded))
        return {"step": state.get("step", 0) + 1, "kind": kind.value, "action": action,
                "retries_left": state["retries_left"] - (action in {"retry", "reinvestigate"}),
                "path": [*state.get("path", []), f"{kind.value} → {action}"]}
    graph = StateGraph(FailState)
    graph.add_node("attempt", attempt)
    graph.add_edge(START, "attempt")
    graph.add_conditional_edges("attempt", lambda s: "attempt" if s["action"] in {"retry", "reinvestigate"} else END, ["attempt", END])
    return graph.compile(), {"retries_left": retries}

print("상태 칸:", list(FailState.__annotations__), "| 모델 호출 0회")

In [ ]:
INJECT = {"pass": (None, FinalAnswer(outcome="answered", text="(주입)", evidence_refs=["F-1"]), []),
          "insufficient": (None, FinalAnswer(outcome="insufficient", text="(주입)"), []),
          "denied": (None, FinalAnswer(outcome="denied", text="(주입)"), []),
          "timeout": (TimeoutError("주입한 시간 초과"), None, []),
          "budget": (BudgetExceeded("주입한 예산 초과"), None, [])}
SCENARIOS = {"근거 부족": ["insufficient", "pass"], "정책 불가": ["denied", "pass"], "시스템 오류": ["timeout", "pass"], "예산 초과": ["budget", "pass"]}

def run_scenarios(decide=next_action, retries: int = 1) -> list[dict]:
    rows = []
    for name, script in SCENARIOS.items():
        app, start = build_failure_graph(lambda step, s=script: INJECT[s[min(step, len(s) - 1)]], decide, retries)
        end = app.invoke(start)
        rows.append({"주입": name, "시도 수": end["step"], "경로": " | ".join(end["path"]), "끝난 행동": end["action"]})
    return rows

show_table(run_scenarios())
print("모델 호출 0회")

In [ ]:
calls = Counter()

def run_policy_graph(node, retry_on=TimeoutError, max_attempts: int = 3) -> str:
    graph = StateGraph(FailState)
    graph.add_node("work", node, retry_policy=RetryPolicy(retry_on=retry_on, max_attempts=max_attempts, initial_interval=0.01, jitter=False))
    graph.add_edge(START, "work")
    graph.add_edge("work", END)
    try:
        return f"끝남: {graph.compile().invoke({'step': 0}).get('kind')}"
    except Exception as error:
        return f"{type(error).__name__}: {str(error)[:50]}"

def recovers(state):
    calls["시간 초과 두 번 뒤 회복"] += 1
    if calls["시간 초과 두 번 뒤 회복"] < 3:
        raise TimeoutError("주입한 시간 초과")
    return {"kind": "pass"}

def always_times_out(state):
    calls["계속 시간 초과"] += 1
    raise TimeoutError("주입한 시간 초과")

def insufficient_answer(state):
    calls["근거 부족을 정상 값으로 반환"] += 1
    return {"kind": "insufficient"}

outcomes = {"시간 초과 두 번 뒤 회복": run_policy_graph(recovers), "계속 시간 초과": run_policy_graph(always_times_out),
            "근거 부족을 정상 값으로 반환": run_policy_graph(insufficient_answer)}
show_table([{"실험": name, "노드 실행 횟수": calls[name], "결과": outcomes[name]} for name in outcomes])
print("모델 호출 0회")

In [ ]:
send_graph = StateGraph(FailState)
send_graph.add_node("work", lambda state: {"kind": "pass"})
send_graph.add_conditional_edges(START, lambda state: [Send("work", {"step": 0}, timeout=0.2)], ["work"])
send_graph.add_edge("work", END)
try:
    print(send_graph.compile().invoke({"step": 0}))
except ValueError as error:
    print("Send(timeout=)을 동기 노드에 쓰면:", type(error).__name__, "-", error)
print("모델 호출 0회")

`RetryPolicy`와 `next_action`은 경쟁 관계가 아니라 맡는 범위가 다릅니다. `RetryPolicy`는 **같은 노드**를 **예외가 났을 때** 다시 실행합니다. 위 표에서 근거 부족을 정상 값으로 돌려준 노드는 한 번만 돕니다. 예외가 아니기 때문입니다. 재조사, 미실행 종료, 사람에게 넘기기는 다른 노드로 가는 일이므로 그래프의 조건부 간선이 맡아야 합니다.

`Send(timeout=)`은 동기 노드에서 `ValueError`가 납니다. 동기 Python 코드는 프로세스 안에서 안전하게 취소할 수 없기 때문입니다. 이 과정의 노드는 동기이므로 시간 제한은 `Budget`이 맡습니다. `Budget`의 시간 한도는 다음 소모 시점에 검사하는 방식이라 이미 돌고 있는 호출을 끊지는 않습니다. 개별 호출의 제한 시간은 `make_model`의 `timeout`이 맡습니다.

이제 실제 Worker로 같은 복구를 합니다. 합류 노드가 `join_results`로 상태를 읽고, `next_action`이 `retry`라고 할 때만 실패한 작업을 새 키 `outage#2`로 다시 보냅니다. `lookup_availability`에는 한 번만 시간 초과를 주입합니다. 이 그래프는 작성자 없이 Worker까지만 돕니다. 복구 경로에만 집중하기 위해서입니다.

In [ ]:
announce("시간 초과 1회 주입 뒤 Worker 복구", model_calls=12)
fault_desk, fault_meter = make_desk(case, faults={"lookup_availability": "timeout_once"}, label="fault"), ModelMeter()

class RecoverState(TypedDict, total=False):
    results: Annotated[dict, merge_results]
    task: dict

def recover_worker(state: RecoverState) -> dict:
    task = WorkerTask.model_validate(state["task"])
    return {"results": {task.task_id: run_worker(task, fault_desk, model, config={"callbacks": [fault_meter]}).model_dump()}}

def after_recover_join(state: RecoverState):
    results = {k: WorkerResult.model_validate(v) for k, v in state["results"].items()}
    info = join_results(results, list(ROLES))
    retries_left = 0 if any("#" in k for k in results) else 1
    action = next_action(FailureKind.SYSTEM_ERROR, retries_left=retries_left, budget_left=True) if info["failed"] else "respond"
    fault_desk.trace.event("join", "code", failed=info["failed"], action=action)
    if action != "retry":
        return END
    return [Send("worker", {"task": WorkerTask(task_id=f"{t}#2", attempt=2, goal=f"{ROLE_GOALS[t]}\n고객 요청: {case['question']}").model_dump()}) for t in info["failed"]]

recover_graph = StateGraph(RecoverState)
recover_graph.add_node("worker", recover_worker)
recover_graph.add_node("join", lambda state: {"results": {}})
recover_graph.add_conditional_edges(START, lambda state: [Send("worker", {"task": WorkerTask(task_id=r, goal=f"{ROLE_GOALS[r]}\n고객 요청: {case['question']}").model_dump()}) for r in ROLES], ["worker"])
recover_graph.add_edge("worker", "join")
recover_graph.add_conditional_edges("join", after_recover_join, ["worker", END])
recovered = recover_graph.compile().invoke({"results": {}}, config={"recursion_limit": 20})
print("모델 호출:", fault_meter.totals()["model_calls"], "| 결과 키:", sorted(recovered["results"]))

In [ ]:
fields = ("at", "kind", "actor", "fault", "failed", "action", "month", "invoice_id", "query")
show_table([{k: e.get(k, "") for k in fields} for e in fault_desk.trace.events])
tool_counts = Counter(e["name"] for e in fault_desk.budget.report()["events"] if e["event"] == "tool")
worker_runs = Counter(s["actor"] for s in fault_desk.trace.spans)
show_table([{"구분": "도구", "이름": name, "횟수": count} for name, count in sorted(tool_counts.items())] +
           [{"구분": "Worker 구간", "이름": name, "횟수": count} for name, count in sorted(worker_runs.items())])
print(timeline(fault_desk.trace.spans))
show_table(join_results({k: WorkerResult.model_validate(v) for k, v in recovered["results"].items()}, list(ROLES))["rows"])

In [ ]:
always_desk = make_desk(case, faults={"lookup_availability": "timeout_always"}, label="always")
availability = always_desk.tools("outage")[0]

def call_availability(step: int) -> tuple:
    # 실제 도구를 부릅니다. 모델은 없습니다. 제어 경로만 확인합니다.
    try:
        availability.invoke({"month": "2026-08"})
    except TimeoutError as error:
        return error, None, []
    return None, FinalAnswer(outcome="answered", text="(조회됨)", evidence_refs=["F-AV-2026-08"]), []

always_app, always_start = build_failure_graph(call_availability)
always_end = always_app.invoke(always_start)
show_table([{"주입": "timeout_always", "시도 수": always_end["step"], "경로": " | ".join(always_end["path"]), "끝난 행동": always_end["action"]}])
print("주입 기록:", Counter(e["kind"] for e in always_desk.trace.events), "| 모델 호출 0회")

**관찰할 것.** 복구의 비용과 보존의 효과를 함께 읽으세요. 사건 표에서 `fault_injected` 다음에 `join`의 `action`이 무엇인지 찾습니다. `Worker 구간` 표에 `outage`와 `outage#2`가 따로 있으면 재시도가 돈 것입니다. `billing`과 `policy`가 각각 1회면 완료된 Worker는 다시 돌지 않았습니다. 재시도의 비용은 `outage#2` 구간과 그만큼의 모델 호출이고, 그 대신 다른 Worker의 호출을 다시 쓰지 않습니다. `outage#2`가 없다면 모델이 도구를 부르지 않았거나 다른 경로로 끝난 것입니다. `lookup_availability` 호출 횟수와 맨 아래 상태 표로 어느 쪽인지 확인하세요.

두 표의 수가 다를 수 있습니다. `도구` 행은 시작한 호출을 세므로 주입으로 실패한 호출까지 포함하고, 사건 표의 `tool` 행은 주입을 지나간 호출만 남습니다. 모델이 한 메시지에서 도구를 둘 불렀다면 하나만 실패해도 그 Worker는 `failed`가 됩니다. 함께 성공한 조회 결과도 Worker 결과로는 돌아오지 않습니다. `month` 칸에서 모델이 어느 달을 물었는지도 보세요. 이 값은 모델이 정합니다. 맨 아래 표에서 `outage`의 상태가 `uncertain`일 수 있습니다. 복구는 실행이 끝났다는 뜻이지 작업이 답을 다 찾았다는 뜻이 아닙니다.

`timeout_always` 행은 `escalate`로 끝나야 합니다. 재시도 기회를 다 쓰면 다시 호출하지 않고 사람에게 넘깁니다. 이것은 제어 경로의 확인이지 장애율 측정이 아닙니다. 주입한 오류는 실제 장애가 얼마나 자주 나는지 아무것도 말해 주지 않습니다.

## 6. 부분 실패와 합류

**문제.** 병렬 Worker 셋 중 하나만 실패했습니다. 그래프는 예외 없이 다음 노드로 갑니다. 합류 노드는 무엇을 해야 할까요? 03에서 만든 병렬은 모두 성공한다고 가정했습니다. 운영에서는 일부만 끝나는 경우가 더 어렵습니다. 전체가 실패하면 누구나 알아차리지만 일부만 실패하면 정상처럼 보일 수 있기 때문입니다.

**먼저 예측하세요.** `outage` Worker가 실패했는데 합류가 성공한 결과만 모아 작성 노드로 넘깁니다. 작성 노드의 입력에 실패를 알 수 있는 흔적이 남을까요? 그 입력으로 쓴 답은 고객에게 어떻게 보일까요? 가용률이 없는 이유를 작성자가 알 수 있을까요?

이 절의 Worker는 일반 함수이고 정해 둔 결과를 돌려줍니다. 합류의 제어를 보는 것이 목적이므로 모델이 필요 없습니다.

In [ ]:
# 실패 재현: 성공한 결과만 모아 넘기는 합류는 실패를 지웁니다.
SAMPLE = {"billing": {"status": "complete", "facts": ["P-9002 1,200,000원은 P-9001과 같은 청구서의 나중 결제입니다."], "evidence_refs": ["F-P-9002"]},
          "policy": {"status": "complete", "facts": ["99.0% 이상 99.5% 미만 구간의 크레딧은 15%입니다."], "evidence_refs": ["E-sample"]},
          "outage": {"status": "failed", "error": "TimeoutError"},
          "outage#2": {"status": "complete", "facts": ["2026-08 가용률은 99.3입니다."], "evidence_refs": ["F-AV-2026-08"]}}
worker_counts = Counter()

class JoinState(TypedDict, total=False):
    results: Annotated[dict, merge_results]
    task: str
    writer_input: dict

def sample_worker(state: JoinState) -> dict:
    task_id = state["task"]
    worker_counts[task_id] += 1
    return {"results": {task_id: WorkerResult(task_id=task_id, **SAMPLE[task_id]).model_dump()}}

def silent_join(state: JoinState) -> dict:
    return {"writer_input": {k: v["facts"] for k, v in state["results"].items() if v["status"] == "complete"}}

silent = StateGraph(JoinState)
silent.add_node("worker", sample_worker)
silent.add_node("join", silent_join)
silent.add_conditional_edges(START, lambda state: [Send("worker", {"task": r}) for r in ROLES], ["worker"])
silent.add_edge("worker", "join")
silent.add_edge("join", END)
silent_end = silent.compile().invoke({"results": {}})
print("작성 노드의 입력:", json.dumps(silent_end["writer_input"], ensure_ascii=False))
print("입력에 outage 언급:", "outage" in json.dumps(silent_end["writer_input"]), "| 예외 발생: 없음 | 모델 호출 0회")

In [ ]:
partial = {"billing": WorkerResult(task_id="billing", **SAMPLE["billing"]),
           "policy": WorkerResult(task_id="policy", status="uncertain", uncertainty=["추가 약정을 찾지 못했습니다."]),
           "outage": WorkerResult(task_id="outage", status="failed", error="TimeoutError")}
required = ["billing", "policy", "outage", "credit"]  # credit은 결과가 아예 오지 않은 작업입니다.
info = join_results(partial, required)
answers = {"complete": (None, FinalAnswer(outcome="answered", text="(주입)", evidence_refs=["F-1"]), []),
           "uncertain": (None, FinalAnswer(outcome="insufficient", text="(주입)"), []),
           "failed": (TimeoutError("주입"), None, []), "결과 없음": (None, None, [])}
rows = []
for row in info["rows"]:
    kind = classify_failure(*answers[row["상태"]])
    rows.append({**row, "실패 종류": kind.value, "다음 행동": next_action(kind, retries_left=1, budget_left=True)})
show_table(rows)
print("ready:", info["ready"], "| failed:", info["failed"], "| uncertain:", info["uncertain"], "| missing:", info["missing"], "| 모델 호출 0회")

In [ ]:
worker_counts.clear()

def visible_join(state: JoinState) -> dict:
    results = {k: WorkerResult.model_validate(v) for k, v in state["results"].items()}
    return {"writer_input": join_results(results, list(ROLES))}

def after_visible_join(state: JoinState):
    failed = state["writer_input"]["failed"]
    return [Send("worker", {"task": f"{t}#2"}) for t in failed] if failed else END

visible = StateGraph(JoinState)
visible.add_node("worker", sample_worker)
visible.add_node("join", visible_join)
visible.add_conditional_edges(START, lambda state: [Send("worker", {"task": r}) for r in ROLES], ["worker"])
visible.add_edge("worker", "join")
visible.add_conditional_edges("join", after_visible_join, ["worker", END])
visible_end = visible.compile().invoke({"results": {}})
show_table([{"작업 키": k, "실행 횟수": n} for k, n in sorted(worker_counts.items())])
show_table(visible_end["writer_input"]["rows"])

first = {"outage": WorkerResult(task_id="outage", **SAMPLE["outage"]).model_dump()}
print("같은 값의 재전달:", sorted(merge_results(first, first)))
try:
    merge_results(first, {"outage": WorkerResult(task_id="outage", **SAMPLE["outage#2"]).model_dump()})
except ValueError as error:
    print("같은 키에 다른 결과:", type(error).__name__, "-", error)
print("모델 호출 0회")

**관찰할 것.** 재현 셀의 출력은 정상 완료와 구별되지 않습니다. 예외도 경고도 없고 작성 노드의 입력에 `outage`라는 글자조차 없습니다. 작성자는 가용률이 왜 없는지 알 수 없으니, 빠뜨린 채 답하거나 다른 값으로 채우게 됩니다. 조용한 합류가 실패 자체보다 위험한 이유가 여기 있습니다. 실패 자체는 재시도할 수 있지만 조용한 합류는 재시도해야 한다는 사실을 지웁니다.

`join_results` 표에서는 네 상태가 다른 행으로 나옵니다. `failed`와 `결과 없음`은 시스템 오류로, `uncertain`은 근거 부족으로 이어집니다. 다음 행동이 다릅니다. 이렇게 누락을 드러내는 책임은 합류 노드에 있습니다. 실행 횟수 표에서 `billing`, `policy`가 각각 1회이고 `outage`와 `outage#2`가 따로 있는지 확인하세요. 재시도는 새 키로 쓰고 `join_results`는 같은 작업의 마지막 시도를 읽으므로, 실패 기록은 남기고 상태 표는 최신으로 읽힙니다. `merge_results`는 같은 키에 다른 값이 오면 조용히 덮어쓰지 않고 멈춥니다.

한계도 있습니다. 이 구조는 합류 시점에 도착한 결과를 읽을 뿐, 동시에 도는 두 시도 중 늦게 도착한 쪽까지 판별하지는 않습니다. 첫 시도가 시간 초과로 처리된 뒤 뒤늦게 응답이 오는 시스템이라면 어떤 값을 더 기록해야 할까요? 시도 번호, 시작 시각, 작업 범위를 무엇으로 묶을지 적어 보세요.

04a의 3절 `검증 문제` 칸에 나온 잘못된 참조도 같은 책임으로 이어집니다. `join_results`는 상태만 읽고 `validate_result`를 부르지 않습니다. 합류에서 근거까지 함께 검사한다면 그 문제를 어떤 실패 종류로 분류하고 어떤 다음 행동으로 이어야 할까요?

### 그림으로 보기: 합류가 실패를 보이게 하는가

위 두 셀의 그래프를 나란히 놓았습니다. 두 그래프 모두 worker 셋 중 `outage`가 실패한 같은 입력을 받습니다.

![위는 silent_join이 complete인 결과만 모아 작성 노드에 넘겨 outage가 입력에서 사라지는 흐름이고, 아래는 join_results가 작업별 상태를 드러내고 failed인 outage만 새 키로 다시 Send해 billing과 policy 결과를 그대로 두는 흐름이다](./assets/04b-2-join-visibility.png)

- 위쪽은 예외도 경고도 없이 끝납니다. 작성 노드의 입력에 `outage`라는 글자조차 없으므로 가용률이 없는 이유를 작성자가 알 수 없습니다.
- 아래쪽은 상태가 `failed`인 작업만 `outage#2`로 다시 보냅니다. 완료된 결과는 다시 돌리지 않습니다.

## 7. 승인과 멱등

**문제.** 지금까지의 실패는 다시 하면 그만이었습니다. 환불은 다릅니다. 1,200,000원을 두 번 돌려주면 다시 돌이키기 어렵습니다. T-1001의 P-9002는 100만원을 초과하므로 재무 담당자 승인이 필요합니다(`billing-policy.md`). 그 흐름은 다음 순서입니다.

처리 제안 → 정책·권한 검증 → 필요한 승인 → 쓰기 실행 → 결과 확인

모델은 **제안**만 합니다. 실행 여부는 코드 게이트와 사람이 정합니다. LangGraph의 `interrupt`는 노드 실행을 멈추고 값을 밖으로 보내며, `Command(resume=...)`로 재개합니다. 이때 재개가 어떻게 동작하는지가 결제 실행의 안전을 좌우합니다.

**먼저 예측하세요.** 노드 안에서 `interrupt` 앞에 원장 호출이 있습니다. 재개하면 노드의 어디부터 다시 실행될까요? `interrupt` 바로 다음 줄일까요, 노드의 첫 줄일까요? 재개까지 끝난 뒤 원장에는 몇 건이 남을까요? 답을 적은 뒤 실행하세요.

In [ ]:
# 실패 재현: 원장 호출을 interrupt 앞에 둔 노드를 재개합니다.
bad_ledger, bad_entries = PaymentLedger(), Counter()

class BadState(TypedDict, total=False):
    request: str
    answer: dict

def refund_then_ask(state: BadState) -> dict:
    bad_entries["노드 첫 줄"] += 1
    bad_ledger.refund("P-9002", 1_200_000)  # 부작용이 interrupt 앞에 있습니다. 멱등 키도 없습니다.
    answer = interrupt({"question": "P-9002 1,200,000원 환불을 승인합니까?"})
    bad_entries["interrupt 뒤"] += 1
    return {"answer": answer}

bad_graph = StateGraph(BadState)
bad_graph.add_node("refund_then_ask", refund_then_ask)
bad_graph.add_edge(START, "refund_then_ask")
bad_graph.add_edge("refund_then_ask", END)
bad_app, bad_config = bad_graph.compile(checkpointer=MemorySaver()), {"configurable": {"thread_id": "bad-1"}}
rows = []
bad_app.invoke({"request": "T-1001"}, config=bad_config)
rows.append({"시점": "승인 대기", "노드 첫 줄": bad_entries["노드 첫 줄"], "interrupt 뒤": bad_entries["interrupt 뒤"], "원장 건수": len(bad_ledger.refunds)})
bad_app.invoke(Command(resume={"approved": True}), config=bad_config)
rows.append({"시점": "승인 재개 후", "노드 첫 줄": bad_entries["노드 첫 줄"], "interrupt 뒤": bad_entries["interrupt 뒤"], "원장 건수": len(bad_ledger.refunds)})
show_table(rows)
show_table(bad_ledger.refunds)
print("승인 전에 이미 돈이 나갔고, 재개가 한 번 더 내보냈습니다. 모델 호출 0회")

In [ ]:
class ApprovalState(TypedDict, total=False):
    question: str
    proposal: dict
    decision: dict
    approved: bool
    execution: dict
    verified: dict
    path: list[str]

def approval_nodes(desk: Desk, propose, request_id: str, lose_response: bool) -> tuple[dict, Counter]:
    entries = Counter()
    def step(state, name):
        entries[name] += 1
        return [*state.get("path", []), name]
    def node_propose(state):  # 제안이 이미 있으면 모델을 부르지 않습니다.
        if state.get("proposal"):
            return {"path": step(state, "propose(입력 사용)")}
        proposal = propose(state["question"])
        return {"proposal": proposal.model_dump(), "path": step(state, "propose")}
    def gate(state):  # 검사만 합니다. 멱등 키를 코드가 정합니다.
        proposal = Proposal.model_validate(state["proposal"])
        proposal = proposal.model_copy(update={"idempotency_key": f"{request_id}:{proposal.payment_id}"})
        decision = check_refund(proposal, desk.payments(), desk.scope.as_of)
        return {"proposal": proposal.model_dump(), "decision": decision.model_dump(), "path": step(state, f"gate:{decision.decision}")}
    def approve(state):  # interrupt 앞에 부작용이 없습니다. 재개하면 이 노드가 처음부터 다시 돕니다.
        path = step(state, "approve")
        answer = interrupt({"proposal": state["proposal"], "reasons": state["decision"]["reasons"]})
        if "amount" in answer:  # 사람이 금액을 고치면 게이트를 다시 지납니다.
            return {"proposal": {**state["proposal"], "amount": answer["amount"]}, "path": path}
        return {"approved": bool(answer.get("approved")), "path": path}
    def execute(state):
        proposal, decision = Proposal.model_validate(state["proposal"]), GateDecision.model_validate(state["decision"])
        try:
            execution = execute_approved(proposal, decision, desk.ledger, approved=bool(state.get("approved")), lose_response=lose_response)
        except ResponseLost as error:
            execution = {"executed": "응답 유실", "error": str(error)}
        return {"execution": execution, "path": step(state, "execute")}
    def verify(state):  # 실행의 응답이 아니라 외부 원장을 다시 읽어 확인합니다.
        record = desk.ledger.find(state["proposal"]["idempotency_key"])
        return {"verified": {"원장에 있음": record is not None, "record": record}, "path": step(state, "verify")}
    return {"propose": node_propose, "gate": gate, "approve": approve, "execute": execute, "verify": verify}, entries

print("상태 칸:", list(ApprovalState.__annotations__), "| 모델 호출 0회")

In [ ]:
def after_gate(state):
    return {"allow": "execute", "needs_approval": "approve"}.get(state["decision"]["decision"], END)

def after_approve(state):
    if "approved" not in state:  # 사람이 금액을 고쳤습니다.
        return "gate"
    return "execute" if state["approved"] else END

def build_approval(desk: Desk, propose=None, *, request_id: str = "T-1001", lose_response: bool = False, with_verify: bool = True):
    nodes, entries = approval_nodes(desk, propose, request_id, lose_response)
    graph = StateGraph(ApprovalState)
    for name, node in nodes.items():
        if name != "verify" or with_verify:
            graph.add_node(name, node)
    graph.add_edge(START, "propose")
    graph.add_edge("propose", "gate")
    graph.add_conditional_edges("gate", after_gate, ["approve", "execute", END])
    graph.add_conditional_edges("approve", after_approve, ["gate", "execute", END])
    graph.add_edge("execute", "verify" if with_verify else END)
    if with_verify:
        graph.add_edge("verify", END)
    return graph.compile(checkpointer=MemorySaver()), entries

show_graph(build_approval(make_desk(case, label="shape"))[0])
print("모델 호출 0회. 그래프를 만드는 것은 호출이 아닙니다.")

### 그림으로 보기: 부작용이 있는 자리

위 셀의 `show_graph`가 그리는 그래프를 부작용이 있는 자리 중심으로 다시 그렸습니다.

![propose 뒤 gate가 검사하고 allow면 execute로, needs_approval이면 approve로 간다. approve는 interrupt로 사람을 기다리고 승인되면 execute로, 금액이 바뀌면 gate로 돌아간다. 원장에 쓰는 곳은 execute뿐이고 verify가 원장을 다시 읽는다](./assets/04b-3-approval-graph.png)

- 원장에 쓰는 노드는 `execute` 하나이고, `approve`에는 부작용이 없습니다. 재개하면 `approve`는 처음부터 다시 돌기 때문입니다.
- `gate`는 검사만 하고 멱등 키를 코드가 정합니다. 사람이 금액을 고치면 `gate`로 돌아가 다시 검사를 받습니다.
- `verify`는 실행의 응답이 아니라 외부 원장을 다시 읽습니다. 응답이 유실돼도 돈이 나갔는지 알 수 있습니다.

In [ ]:
announce("모델이 T-1001 환불 제안을 작성", model_calls=5)
approval_desk, propose_meter = make_desk(case, label="approval"), ModelMeter()
PROPOSE_PROMPT = (f"환불 제안을 작성합니다. {scope_line(approval_desk)} 같은 청구서에 승인된 결제가 두 건 이상이면 중복 결제입니다. "
                  "먼저 승인된 결제는 유효 결제이고 나중 결제가 환불 대상입니다. 기준일 기준 결제일부터 90일 이내의 중복 결제만 이번 환불 대상입니다. "
                  "결제를 조회한 뒤 한 건을 골라 action=refund, payment_id, 결제 금액 amount, reason을 쓰세요. 직접 실행하지 않습니다.")

def model_propose(question: str) -> Proposal:
    agent = make_worker("billing", approval_desk, model, prompt=PROPOSE_PROMPT, response_format=Proposal)
    state = agent.invoke({"messages": [{"role": "user", "content": question}]}, config={"callbacks": [propose_meter], "recursion_limit": 40})
    return state.get("structured_response") or Proposal(reason="구조화 응답 없음")

approval_app, approval_entries = build_approval(approval_desk, model_propose)
approval_config = {"configurable": {"thread_id": "t-1001"}}
waiting = approval_app.invoke({"question": case["question"]}, config=approval_config)
ledger_before = len(approval_desk.ledger.refunds)
print("모델 호출:", propose_meter.totals()["model_calls"], "| 경로:", waiting["path"])
print("제안:", json.dumps(waiting["proposal"], ensure_ascii=False))
print("게이트:", waiting["decision"])
print("멈춘 곳:", approval_app.get_state(approval_config).next, "| interrupt 값:", [i.value["reasons"] for i in waiting.get("__interrupt__", [])])
print("승인 전 원장 건수:", ledger_before)

In [ ]:
if approval_app.get_state(approval_config).next:
    approved_end = approval_app.invoke(Command(resume={"approved": True}), config=approval_config)
    ledger_after = len(approval_desk.ledger.refunds)
    print("경로:", approved_end["path"])
    print("실행 결과:", approved_end["execution"])
    print("결과 확인:", approved_end["verified"])
    show_table([{"노드": name, "실행 횟수": count} for name, count in approval_entries.items()])
else:
    ledger_after = len(approval_desk.ledger.refunds)
    print("이번 제안은 승인 단계까지 오지 않았습니다. 게이트 결정:", waiting["decision"])
print("승인 전 원장:", ledger_before, "건 | 승인 후 원장:", ledger_after, "건 | 모델 호출 0회")

In [ ]:
FIXED_PROPOSAL = Proposal(action="refund", payment_id="P-9002", amount=1_200_000, reason="T-1001 고정 입력")

def resume_with(thread: str, answer: dict) -> dict:
    # 새 thread와 새 Desk에서 고정 제안으로 시작합니다. 모델을 부르지 않습니다.
    desk = make_desk(case, label=thread)
    app, _ = build_approval(desk)
    config = {"configurable": {"thread_id": thread}}
    app.invoke({"proposal": FIXED_PROPOSAL.model_dump()}, config=config)
    end = app.invoke(Command(resume=answer), config=config)
    return {"재개 값": str(answer), "경로": " → ".join(end["path"]), "게이트 결정": end["decision"]["decision"],
            "게이트 이유": end["decision"]["reasons"][0], "대기 중": bool(app.get_state(config).next), "원장 건수": len(desk.ledger.refunds)}

show_table([resume_with("rejected", {"approved": False}), resume_with("edited", {"approved": True, "amount": 1_000_000})])
print("금액을 고친 재개는 승인으로 바로 가지 않고 게이트를 다시 지납니다. 모델 호출 0회")

In [ ]:
# 실패 재현: 응답이 유실된 뒤 키 없이 재시도하면 두 번 환불되고, 키로 재시도하면 한 번입니다.
def retry_after_lost(key: str) -> dict:
    ledger = PaymentLedger()
    proposal = FIXED_PROPOSAL.model_copy(update={"idempotency_key": key})
    decision = check_refund(proposal, case_desk.payments(), case_desk.scope.as_of)
    try:
        execute_approved(proposal, decision, ledger, approved=True, lose_response=True)
    except ResponseLost as error:
        lost = str(error)
    after_loss = len(ledger.refunds)
    retried = execute_approved(proposal, decision, ledger, approved=True)
    return {"멱등 키": key or "(없음)", "유실 메시지": lost, "유실 직후 원장": after_loss, "재시도 후 원장": len(ledger.refunds),
            "재시도 응답 refund_id": retried["record"]["refund_id"], "원장의 refund_id": [r["refund_id"] for r in ledger.refunds]}

lost_rows = [retry_after_lost(""), retry_after_lost("T-1001:P-9002")]
show_table(lost_rows)
print("키 없이 재시도:", lost_rows[0]["재시도 후 원장"], "건 | 키로 재시도:", lost_rows[1]["재시도 후 원장"], "건 | 모델 호출 0회")

In [ ]:
def recover(proposal: Proposal, decision: GateDecision, ledger: PaymentLedger, first_lost: bool) -> list[dict]:
    # 응답이 없으면 먼저 외부 결과를 조회합니다. 있으면 재시도하지 않고, 없을 때만 다시 실행합니다.
    steps = []
    try:
        record = execute_approved(proposal, decision, ledger, approved=True, lose_response=first_lost)["record"]
        steps.append({"단계": "실행", "결과": "응답 받음", "refund_id": record["refund_id"]})
    except ResponseLost as error:
        steps.append({"단계": "실행", "결과": f"응답 유실: {error}", "refund_id": ""})
        found = ledger.find(proposal.idempotency_key)
        steps.append({"단계": "ledger.find", "결과": "기록 있음. 재시도하지 않습니다." if found else "기록 없음. 다시 실행합니다.", "refund_id": found["refund_id"] if found else ""})
    steps.append({"단계": "원장 건수", "결과": len(ledger.refunds), "refund_id": ""})
    return steps

keyed = FIXED_PROPOSAL.model_copy(update={"idempotency_key": "T-1001:P-9002"})
keyed_decision = check_refund(keyed, case_desk.payments(), case_desk.scope.as_of)
show_table(recover(keyed, keyed_decision, PaymentLedger(), first_lost=True))
print("모델 호출 0회")

**관찰할 것.** 재현 셀의 표에서 `노드 첫 줄` 칸과 `interrupt 뒤` 칸을 비교하세요. 재개는 멈춘 줄에서 이어지는 것이 아니라 노드를 처음부터 다시 실행합니다. 그래서 승인 대기 시점에 이미 원장에 1건이 있고, 재개가 끝나면 2건이 됩니다. 사람이 승인하기도 전에 돈이 나갔다는 점이 두 번 나간 것만큼 심각합니다. 올바른 그래프의 `실행 횟수` 표에서 `approve`가 `execute`보다 많은 것을 찾으세요. `approve` 노드도 재개 때 처음부터 다시 돕니다. 그 앞에 부작용이 없어 해가 없을 뿐입니다.

제안 셀은 모델 결과에 따라 달라집니다. `gate:needs_approval`로 멈췄다면 `멈춘 곳`이 `approve`이고 원장은 0건이어야 합니다. 모델이 다른 결제나 다른 금액을 제안했다면 게이트가 거절해 멈추지 않습니다. 그때는 승인 칸이 비고, 거절 이유가 곧 관찰 결과입니다. 어느 쪽이든 모델의 제안만으로 원장이 바뀌는 경로는 없어야 합니다. 거절 행과 금액 수정 행의 `원장 건수`, 그리고 금액 수정 행의 `게이트 이유`를 확인하세요. 금액을 고치면 승인했더라도 게이트를 다시 지납니다. 이것이 승인의 전부는 아닙니다. 승인한 내용과 실제 실행 인자를 묶는 승인 바인딩은 Day08에서 다룹니다.

응답 유실 표는 `재시도 후 원장` 칸을 읽습니다. 원장은 기록을 남긴 뒤 응답만 잃었습니다. 키 없는 재시도는 원장이 같은 요청인지 알 수 없어 새 환불을 만들고, 키가 있으면 같은 `refund_id`를 돌려줍니다. 그다음 셀은 더 나은 순서입니다. 응답이 없으면 곧바로 다시 부르지 않고 `ledger.find`로 외부 결과부터 조회합니다.

체크포인트는 우리 그래프의 진행을 저장할 뿐입니다. `MemorySaver`가 있어도 첫 번째 재현에서 두 건이 남았습니다. `idempotency_key`는 외부 시스템이 지원하고 강제해야 뜻이 있으며, 재개한다고 exactly-once가 자동으로 보장되지 않습니다. 여기의 `PaymentLedger`는 그 강제를 흉내 낸 가짜입니다. 실제 결제 API가 같은 키 처리, 같은 보관 기간, 같은 조회 기능을 제공한다고 가정하면 안 됩니다. 실제 연동 전에 그 API의 문서로 확인해야 합니다.

## 8. 완성형 조립: Workflow 안에 필요한 Agent만 배치합니다

**문제.** 앞 절의 부품을 한 그래프에 모으면 어떤 모양이 될까요? 전체를 Agent 하나에게 맡기지도, 전체를 고정 코드로 짜지도 않습니다. 코드가 경로와 쓰기를 정하고, 판단이 필요한 노드에서만 모델과 Agent를 씁니다. 라우팅, 조사, 작성은 모델이 맡습니다. 범위 확인, 합류, 검증, 게이트, 승인, 실행은 코드가 맡습니다.

소스를 읽으며 다음을 찾아보세요. `gate` 노드에 왜 부작용이 없을까요? `after_join`은 어떤 조건에서 `retry`로 갈까요? `execute`는 어떤 함수를 통해 원장에 닿을까요? 아래 표는 각 노드를 04a 1절의 층에 대응시킵니다. 그림의 점선은 조건부 간선인데 어떤 조건에서 어느 갈래로 가는지는 그림에 적혀 있지 않으므로, 갈림 조건은 아래 셀이 출력하는 `build_hybrid` 소스에서 찾거나 그림 아래 간선 표의 `조건부` 열이 `True`인 행을 찾아 그 간선을 고르는 라우팅 함수에서 읽어야 합니다.

In [ ]:
print(inspect.getsource(build_hybrid))
hybrid_shape = build_hybrid(make_desk(case, label="shape"), model)  # 만들기만 합니다. 호출이 아닙니다.
show_graph(hybrid_shape)
NODE_LAYERS = {"scope": ("Harness", "코드", "범위 밖 고객을 멈춥니다"), "router": ("Prompt, Graph", "모델", "경로와 조사 범위를 고릅니다"),
               "fast": ("Prompt, Context", "Agent", "단순 질문을 바로 답합니다"), "worker": ("Context", "Agent", "WorkerTask를 받아 WorkerResult를 돌려줍니다"),
               "join": ("Graph", "코드", "결과를 합치고 상태를 읽습니다"), "retry": ("Loop", "Agent", "실패한 작업만 새 키로 다시 돌립니다"),
               "writer": ("Context", "Agent", "결과와 calculate_credit으로 답을 씁니다"), "validate": ("Harness, Loop", "코드", "근거와 숫자를 검사하고 재작성합니다"),
               "gate": ("Harness", "코드", "게이트 판단으로 경로를 나눕니다"), "approve": ("Harness", "사람", "interrupt로 승인을 기다립니다"), "execute": ("Harness", "코드", "gate_answer로만 원장에 닿습니다")}
show_table([{"노드": node, "층": layer, "누가 하는가": actor, "역할": role} for node, (layer, actor, role) in NODE_LAYERS.items()])
print("모델 호출 0회")

In [ ]:
announce("완성형 조립으로 T-1001 처리", model_calls=16)
hybrid_row = run_case("hybrid", case, model=model)  # approval이 None이면 승인 대기로 끝납니다.
print("status:", hybrid_row["status"], "| outcome:", hybrid_row["outcome"], "| 모델 호출:", hybrid_row["model_calls"], "| 시간:", hybrid_row["elapsed_seconds"], "초")
print(hybrid_row["answer"])
print("불확실성:", hybrid_row.get("uncertainty"))

In [ ]:
show_table([{k: hybrid_row[k] for k in ("structure", "status", "outcome", "control_owner", "final_speaker", "model_calls", "tool_calls", "input_tokens", "output_tokens",
                                        "context_bytes", "max_worker_overlap_seconds", "ledger_refunds", "policy_violations")}])
print(timeline(hybrid_row["spans"]))
hybrid_check = evaluate_row(hybrid_row, gold)
show_table([{k: hybrid_check[k] for k in ("자동 검사", "기대 outcome", "실제 outcome", "누락 사실", "금지 포함", "누락 문서", "원장 건수", "실패 칸")}])
print("제안:", hybrid_row.get("proposal"))
print("인용 문서:", hybrid_row["cited_docs"], "| 도구 호출:", Counter(hybrid_row["tools_called"]))
print("검증 사건:", [e["problems"] for e in hybrid_row["events"] if e["kind"] == "validate"])
print("오류:", hybrid_row["errors"] or "없음", "| usage 누락:", hybrid_row["usage_missing"], "| 이 셀의 모델 호출 0회")

**관찰할 것.** 처음의 업무 목표를 끝까지 추적하세요. 고객의 요청은 두 가지였습니다. 중복 결제의 취소와 보상안 확인입니다. `answer`에 환불 대상과 금액, 가용률, 크레딧 비율과 금액, 신청 기한이 있는지 읽고 `자동 검사` 칸과 비교합니다. 실패 칸이 있으면 어느 층의 문제인지 04a의 1절 표로 돌아가 말해 보세요. 필수 사실이 없다면 Context의 문제일 수 있고, 잘못된 금액이 있다면 도구를 쓰지 않은 Prompt나 Harness의 문제일 수 있습니다. 자동 검사가 통과했더라도 원문과 대조하는 일은 남습니다. 반대로 `실패 칸`이 `필수 사실` 하나이고 `누락 사실`이 `9월20일`이라면, 먼저 답에 `2026-09-20`처럼 다른 표기가 있는지 찾으세요. 있다면 답이 아니라 자동 검사의 한계입니다.

`timeline`에 `writer`가 두 줄이면 재작성이 일어난 것입니다. `검증 사건` 줄의 첫 목록이 비어 있지 않으면 `validate` 노드가 문제를 찾아 작성 노드로 돌려보낸 것이고, 둘째 목록이 비어 있으면 재작성이 문제를 해소한 것입니다. 이 재작성에 쓴 호출도 `model_calls`에 들어갑니다. 재작성 없이 끝났다면 목록이 하나입니다.

실행과 미실행의 결과까지 설명할 수 있어야 합니다. `outcome`이 `needs_approval`이고 `ledger_refunds`가 0이면 환불은 아직 일어나지 않았습니다. 답의 문장이 "환불했습니다"처럼 읽힌다면 `policy_violations` 칸과 함께 살펴보세요. 이 칸은 outcome과 원장을 대조할 뿐, 문장의 시제까지 보지는 않습니다. `timeline`에서 Worker 구간이 세로로 겹치는지도 보세요. 이 값 역시 1회 관찰입니다. 05에서 같은 조건으로 구조를 비교할 때 다시 쓰게 됩니다.

## 직접 해보기

두 칸 모두 모델을 부르지 않고 바로 실행됩니다. 조건 하나만 바꾸세요. 먼저 무엇이 달라질지 적은 뒤 돌립니다. 계약과 전달물을 바꿔 보는 해보기는 04a에 있습니다.

### 해보기 1

`next_action`을 고쳐 정책 불가도 재조사하게 합니다. 5절의 주입 그래프에서 경로가 어떻게 바뀌는지 보세요. 재조사를 세 번 허용하면 무엇이 늘어나고, 그것이 고객에게 어떤 뜻인지 말해 보세요. `정책 불가` 행의 `시도 수`가 달라지는데, 두 번째 시도에서 통과로 끝났다면 그 통과를 믿어도 될까요? 주입 대본이 그렇게 짜여 있기 때문인지, 재조사가 규정을 바꿀 수 있어서인지 구분해 답하세요.

In [ ]:
RETRIES = 3

def reinvestigate_policy(kind: FailureKind, *, retries_left: int, budget_left: bool) -> str:
    if kind is FailureKind.POLICY_DENIED:
        return "reinvestigate" if retries_left > 0 and budget_left else "stop_with_reason"
    return next_action(kind, retries_left=retries_left, budget_left=budget_left)

SCENARIOS["정책 불가(계속 거절)"] = ["denied"]
show_table(run_scenarios(reinvestigate_policy, RETRIES))
show_table(run_scenarios(next_action, RETRIES))
del SCENARIOS["정책 불가(계속 거절)"]
print("모델 호출 0회")

### 해보기 2

승인 그래프에서 `verify`를 빼고 응답 유실을 넣습니다. 두 행의 `실행 결과`와 `결과 확인`, `원장 건수`를 비교하세요. `verify`가 없으면 돈이 나갔는지 그래프 상태만으로 알 수 있습니까? 이때 사람이 "실패했으니 다시 실행"을 누르면 어떤 일이 생길까요?

In [ ]:
WITH_VERIFY = [True, False]
rows = []
for with_verify in WITH_VERIFY:
    desk = make_desk(case, label="exercise-4")
    app, _ = build_approval(desk, lose_response=True, with_verify=with_verify)
    config = {"configurable": {"thread_id": f"lost-{with_verify}"}}
    app.invoke({"proposal": FIXED_PROPOSAL.model_dump()}, config=config)
    end = app.invoke(Command(resume={"approved": True}), config=config)
    rows.append({"verify": with_verify, "경로": " → ".join(end["path"]), "실행 결과": end["execution"].get("executed"),
                 "결과 확인": end.get("verified", {}).get("원장에 있음", "알 수 없음"), "원장 건수": len(desk.ledger.refunds)})
show_table(rows)
print("모델 호출 0회")

In [ ]:
# 아래 답을 직접 바꿔 쓰세요. 06이 이 기록을 모아 읽습니다.
ANSWER = ("무엇을 보고 무엇을 실행할 수 있는가: (내 업무의 Worker마다 볼 수 있는 자료와 쓸 수 있는 도구, 실행은 누가 하는가)\n"
          "언제 재시도하고 언제 전환하는가: (어떤 실패에 재시도하고, 어떤 실패에 사람에게 넘기거나 멈추는가)\n"
          "예산과 종료: (한 실행의 모델 호출 상한, 그 근거, 넘었을 때 무엇을 남기는가)")
path = write_design_log("Context·권한과 예산·종료: 무엇을 보고 무엇을 실행하며 언제 멈출까?", ANSWER)
print("설계 기록을 썼습니다:", path.relative_to(Path.cwd().resolve()))

## 정리

이 노트북에서 그래프에 넣은 것을 다시 적습니다. 04a의 Prompt, Context, Harness 정리와 이어서 읽으세요.

- **Graph.** 합류 노드가 필수 작업별 상태를 드러냅니다. 조용히 빠뜨리지 않습니다.
- **Loop.** 실패의 종류가 다음 행동을 정합니다. 재시도, 재조사, 멈춤, 사람에게 넘기기는 서로 대체할 수 없습니다.
- **승인과 멱등.** 부작용은 `interrupt` 뒤의 별도 노드에 둡니다. 재개는 노드를 처음부터 다시 돌립니다. 체크포인트는 그래프의 진행만 저장하고, 중복 실행을 막는 일은 외부 시스템의 멱등 키에 달려 있습니다.

**Day08에서 이어지는 항목.** 승인 기록과 실행 인자를 묶는 승인 바인딩(도구, 인자 해시, 만료)과 재검증, 입력과 출력의 Guardrail, 권한과 메모리 방어를 다룹니다.

## 확인 질문

1. 시스템 오류와 근거 부족의 다음 행동은 왜 달라야 합니까?
2. 실패한 Worker를 조용히 빠뜨린 합류가 왜 그 실패 자체보다 위험합니까?
3. `MemorySaver`가 있는데 왜 `idempotency_key`가 필요합니까? 7절의 첫 번째 재현과 응답 유실 재현의 차이는 무엇입니까?
4. 여기 원장의 멱등 보장을 실제 결제 API에 그대로 기대할 수 없는 이유는 무엇입니까?

다음 05에서는 이 노트북의 부품을 고정한 채 구조만 바꿉니다. 실패 분류에서 출발해 같은 조건에서 구조를 재고, 나누는 이유와 나누지 않는 이유를 숫자로 뒷받침합니다.